# Lesson 5: Fine-tune our first intent model

**Run 1 = reproduce the paper.** The ArBanking77 authors fine-tuned AraBERTv2 on MSA only and got **macro F1 = 0.9099** on the MSA test set. We train the same model, on the same data, with their settings, and check that we get about the same number.

If we match it, our training setup is trustworthy, and every later experiment (MARBERT, English, Egyptian data) can be compared against it.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

## 1. Check the GPU
Training on a GPU takes minutes; on a normal CPU it would take hours.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Get our project into Colab
Colab is a fresh computer in the cloud, so we copy our GitHub repository into it, install one missing library, and rebuild the data with notebook 02 (the `data/` folder is not on GitHub, on purpose).

In [ ]:
!test -d banking-assistant || git clone https://github.com/YoussefZaky208/banking-assistant.git
%cd /content/banking-assistant
!pip install -q datasets
!cd notebooks && python 02_load_and_align_data.py

## 3. The experiment settings
Everything that defines this run is in one dictionary. To run a different experiment later, we only change values here.
- **learning rate:** how big each correction step is. Too big = unstable, too small = very slow.
- **batch size:** how many questions the model looks at before each correction.
- **max length:** questions longer than 128 tokens are cut (ours are much shorter).
- **epoch:** one full pass over all training questions.
- **patience:** stop if validation F1 has not improved for 5 epochs (**early stopping**), to avoid **overfitting** (memorizing training data instead of learning).

In [ ]:
# Every setting of the experiment lives here, so each run is easy to describe
# and repeat. Run 1 reproduces the paper: AraBERTv2 trained on MSA only.
CONFIG = {
    "run_name": "run1_arabertv2_msa",
    "model_name": "aubmindlab/bert-base-arabertv02",
    "languages": ["ar"],        # ["ar"] = MSA only (like the paper), ["ar", "en"] = both
    "normalize": False,         # our normalize_arabic; off to match the paper
    "learning_rate": 4e-5,      # paper's best value
    "batch_size": 64,           # paper's best value
    "max_length": 128,          # paper's value
    "max_epochs": 20,           # paper: up to 20 ...
    "patience": 5,              # ... stop after 5 epochs without improvement
    "seed": 42,
}
PAPER_MSA_F1 = 0.9099           # paper, AraBERTv2 fine-tuned on MSA only

## 4. Load the data
We keep only the languages in the config, and turn each intent name into its number (0-76) from `labels.json`.

In [ ]:
import json
import sys

import numpy as np
import pandas as pd

sys.path.append(".")
from src.text_utils import normalize_arabic

data = pd.read_csv("data/banking_en_ar.csv")
label2id = json.load(open("data/labels.json", encoding="utf-8"))
id2label = {i: label for label, i in label2id.items()}

data = data[data["lang"].isin(CONFIG["languages"])].copy()
if CONFIG["normalize"]:
    data["text"] = data["text"].map(normalize_arabic)
data["label"] = data["intent"].map(label2id)

train_df = data[data["split"] == "train"]
val_df = data[data["split"] == "val"]
test_df = data[data["split"] == "test"]
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
print(test_df["variety"].value_counts())

## 5. Tokenize
The model's own tokenizer turns every question into token numbers (Lesson 4). `[CLS]` and `[SEP]` are added at the start and end. Questions are not padded yet: padding happens per batch later (**dynamic padding**), which is faster.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])


def to_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    return ds.map(lambda batch: tokenizer(batch["text"], truncation=True,
                                          max_length=CONFIG["max_length"]),
                  batched=True)


train_ds, val_ds, test_ds = to_dataset(train_df), to_dataset(val_df), to_dataset(test_df)

example = train_ds[0]
print(example["text"])
print(tokenizer.convert_ids_to_tokens(example["input_ids"]))
print("label:", example["label"], "=", id2label[example["label"]])

## 6. Load the model
AraBERTv2 already knows Arabic from pre-training on a huge amount of text. We add a new **classification head**: a small layer that turns the sentence representation into 77 scores, one per intent. Only the head starts from scratch; the rest is fine-tuned.

The message saying some weights were *newly initialized* is expected: that is the new head.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

## 7. How we score
- **accuracy:** share of questions with the correct intent.
- **macro F1:** F1 per intent, averaged, so all 77 intents count equally (Lesson 3). This is the paper's metric and our main one.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

## 8. Set up training
The **Trainer** runs the training loop for us: show a batch, compute the **loss** (how wrong the model was), adjust the weights to reduce it, repeat. After each epoch it checks validation macro F1, keeps the best epoch, and stops early if it stops improving.

In [ ]:
from transformers import (DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)
import torch

set_seed(CONFIG["seed"])

args = TrainingArguments(
    output_dir=f"checkpoints/{CONFIG['run_name']}",
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=128,
    num_train_epochs=CONFIG["max_epochs"],
    eval_strategy="epoch",           # check validation after every epoch
    save_strategy="epoch",
    save_total_limit=1,              # keep disk usage small
    load_best_model_at_end=True,     # end with the best epoch, not the last
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),  # faster on GPU
    logging_steps=50,
    report_to="none",
    seed=CONFIG["seed"],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["patience"])],
)

## 9. Train
This takes about 10-20 minutes on a T4. Watch two things:
- **Training loss** should go down.
- **Validation macro F1** should go up, then flatten. If training loss keeps dropping but validation stops improving, the model has started overfitting, and early stopping ends the run.

In [ ]:
train_result = trainer.train()

## 10. Training history
One row per epoch. Find the epoch with the best validation macro F1: that is the model we keep.

In [ ]:
history = pd.DataFrame([h for h in trainer.state.log_history if "eval_macro_f1" in h])
history[["epoch", "eval_loss", "eval_accuracy", "eval_macro_f1"]].round(4)

## 11. The real test: compare with the paper
The test set was never used for training or for choosing the best epoch, so this is the honest score.

In [ ]:
# Early stopping was only for training; remove it before the final test.
trainer.remove_callback(EarlyStoppingCallback)

results = {}
for variety in sorted(test_df["variety"].unique()):
    part = test_df[test_df["variety"] == variety]
    metrics = trainer.evaluate(to_dataset(part), metric_key_prefix="test")
    results[variety] = {"accuracy": round(metrics["test_accuracy"], 4),
                        "macro_f1": round(metrics["test_macro_f1"], 4),
                        "n": len(part)}
print(pd.DataFrame(results).T)
if "msa" in results:
    print(f"\nMSA macro F1: ours {results['msa']['macro_f1']:.4f} | paper {PAPER_MSA_F1}")

## 12. Which intents are hard?
The overall score hides the details. Here we look at the 10 worst intents, and at the urgent ones a bank cares most about.

In [ ]:
from sklearn.metrics import classification_report

pred = trainer.predict(test_ds)
test_preds = np.argmax(pred.predictions, axis=-1)
report = classification_report(test_ds["label"], test_preds, labels=list(range(len(label2id))),
                               target_names=[id2label[i] for i in range(len(label2id))],
                               output_dict=True, zero_division=0)
per_intent = pd.DataFrame(report).T.iloc[:len(label2id)][["precision", "recall", "f1-score", "support"]]
print("Worst 10 intents:")
print(per_intent.sort_values("f1-score").head(10).round(3))
print("\nUrgent intents:")
print(per_intent.loc[["lost_or_stolen_card", "card_swallowed", "compromised_card"]].round(3))

## 13. Read the mistakes
Numbers tell you how often the model is wrong; reading mistakes tells you why. Often the question is ambiguous, or the two intents are very close.

In [ ]:
test_view = test_df.reset_index(drop=True).assign(
    predicted=[id2label[i] for i in test_preds])
mistakes = test_view[test_view["intent"] != test_view["predicted"]]
print(len(mistakes), "mistakes out of", len(test_view))
mistakes[["text", "intent", "predicted"]].sample(min(15, len(mistakes)), random_state=0)

## 14. Save the results
Colab deletes everything when the session ends, so download the results file and put it in a `results` folder in your project on your laptop.

In [ ]:
import os

os.makedirs("results", exist_ok=True)
summary = {"config": CONFIG, "test": results,
           "best_val_macro_f1": trainer.state.best_metric,
           "epochs_trained": trainer.state.epoch}
with open(f"results/{CONFIG['run_name']}.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


from google.colab import files
files.download(f"results/{CONFIG['run_name']}.json")